# R05 · Cookbook Reference Lab: MATH_MAP

**NHyDB 0.28.2 notebook series · v1 · reference part**

These examples are based on the official
[NHyDB Python Cookbook](https://github.com/nhydb-run4/NHyDB/blob/main/docs/cookbook/PYTHON_COOKBOOK.md) (section 18), which serves as their authoritative reference.
Code and expected output are taken unchanged from the cookbook, and the same examples are run by the
NHyDB test suite.

* Every code cell is an **independent example** – run any cell on its own, or *Run All*.
* Each example creates its own temporary database folder.
* Under each example you find the output printed in the cookbook, to compare with yours.
* The main series (`../main/`) explains the ideas behind these APIs step by step.

**Contents** (6 runnable examples)

* 18. MATH_MAP: placing values by formulas

In [1]:
import nhydb

print('nhydb', nhydb.ENGINE_VERSION, '· file format', nhydb.FILE_FORMAT_VERSION)
if nhydb.ENGINE_VERSION != '0.28.2':
    print('⚠ These examples were written for NHyDB 0.28.2; results may differ.')

nhydb 0.28.2 · file format 0.28.2


## 18. MATH_MAP: placing values by formulas

A MATH layout stores a value at a cell **computed** from input values:
`inputs → (u, V, side)` with integer expressions. `V` is a global
coordinate; the engine maps it to realms automatically and owns these
realms exclusively.

In [2]:
from nhydb.math import MathExpression, MathMap

e = MathExpression.compile("gcd(a, b) + a // 2", ("a", "b"))
print(e.evaluate(a=12, b=18))

grid = MathMap.compile("grid", ("x", "y"), u="x + 1", V="y", side="x % 2")
t = grid.target(3, 5)
print(t.u, t.V_global, t.logical_realm, t.v_local, t.side)
print(grid.target(3, 10**100 + 1).logical_realm)          # V beyond 10^100 -> next logical realm
print([t.u for t in grid.targets([(1, 1), (2, 2)])])
print(MathMap.from_dict(grid.to_dict()).fingerprint == grid.fingerprint)

from nhydb.math import DEFAULT_REALM_AUTO_LIMIT, join_global_v, normalize_side, split_global_v
print(split_global_v(10**100 + 5), join_global_v(2, 5) == 10**100 + 5)
print(normalize_side("NR"), normalize_side(False), DEFAULT_REALM_AUTO_LIMIT)

12
4 5 1 5 1
2
[2, 3]
True
(2, 5) True
1 0 4096


*Expected output (from the cookbook):*

```text
12
4 5 1 5 1
2
[2, 3]
True
(2, 5) True
1 0 4096
```

Expressions use Python syntax for integers: `+ - * // % **`, comparisons,
`and`/`or`/`not` and the functions `abs`, `gcd`, `lcm`, `isqrt`, `min`,
`max`. Not allowed: `/`, bit operators, attributes, conditional
expressions. Errors raise `MathSyntaxError` or `MathDomainError` (both
subclasses of `MathError`, which is an `NHyDBError`). Brackets may be
nested 200 levels deep as in CPython; long `+` chains are fine.

**Logical and physical realms.** `V` is split into a *logical* realm
(`⌊(V−1)/10^100⌋ + 1`, unbounded – 10^5000 is fine) and a local `v`. The
engine stores the cell under the next free *physical* realm (small numbers
from 1 upward) and keeps the mapping in the encrypted layout plan. The
`N` of a stored MATH cell is therefore computed from the physical realm;
`get_math_cell` also reports `_logical_realm` and `_V_global`.
`realm_auto_limit` limits the **number** of logical realms a layout may
use, not their size.

Check a mapping before writing with `dry_run` (no database needed):

In [3]:
from nhydb.math import MathMap, dry_run

grid = MathMap.compile("grid", ("x", "y"), u="x + 1", V="y", side="NL")
report = dry_run(grid, [(1, 1), (1, 1), (2, 10**100 + 3), (-5, 1)])
print(report.total, report.valid, report.invalid, report.collisions, report.unique_cells)
print(report.logical_realms, report.invalid_examples)

4 3 1 1 2
(1, 2) (((-5, 1), 'u must be a positive integer'),)


*Expected output (from the cookbook):*

```text
4 3 1 1 2
(1, 2) (((-5, 1), 'u must be a positive integer'),)
```

Write and read MATH data:

In [4]:
import os
import tempfile

import nhydb
from nhydb.math import MathMap, MathMapLayoutSpec

path = os.path.join(tempfile.mkdtemp(), "math.nhydb")
with nhydb.create(path) as db:
    grid = MathMap.compile("grid", ("x", "y"), u="x + 1", V="y", side="x % 2")
    state = db.create_math_layout(MathMapLayoutSpec("grid", grid, 64), layout_id=7, payload_type="utf8")
    print(state)

    print(db.put_math(7, (1, 7), "first"))                           # one value
    report = db.put_math_batch(7, [((x, y), f"{x},{y}") for x in (2, 3) for y in (1, 10**100 + 1)],
                               commit=True)
    print(report["cells"], report["new_realms"], report["realm_extents"])
    print(db.get_math(7, (3, 10**100 + 1)), db.get_math(7, (9, 9)))
    cell = db.get_math_cell(7, (3, 10**100 + 1))
    print(cell["_realm"], cell["_u"], cell["_lane_name"], cell["_logical_realm"])
    info = db.describe_math(7)
    print(info["frozen"], info["payload_type"], info["mapped_realms"])
    try:
        db.put_math(7, (2, 1), "again")                              # cell already used
    except nhydb.MathCellConflictError:
        print("conflict")
    print(db.put_math(7, (2, 1), "replaced", overwrite=True)["u"], db.get_math(7, (2, 1)))

MathLayoutState(layout_id=7, name='grid', frozen=False, payload_type='utf8', mapped_realms=0)
{'layout_id': 7, 'physical_realm': 1, 'u': 2, 'v': 7, 'lane': 1, 'lane_name': 'NR'}
4 1 [{'logical_start': 1, 'logical_end': 2, 'physical_start': 1, 'physical_end': 2}]
3,10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001 None
2 4 NR 2
True utf8 2
conflict
3 replaced


*Expected output (from the cookbook):*

```text
MathLayoutState(layout_id=7, name='grid', frozen=False, payload_type='utf8', mapped_realms=0)
{'layout_id': 7, 'physical_realm': 1, 'u': 2, 'v': 7, 'lane': 1, 'lane_name': 'NR'}
4 1 [{'logical_start': 1, 'logical_end': 2, 'physical_start': 1, 'physical_end': 2}]
3,10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001 None
2 4 NR 2
True utf8 2
conflict
3 replaced
```

A whole `put_math_batch` is rolled back if any row fails. The short form
`create_math_map_layout` takes the expressions directly; the mapping can be
replaced with `replace_math_spec` only **before** the first write
(afterwards `MappingFrozenError`):

In [5]:
import os
import tempfile

import nhydb
from nhydb.math import MathMap, MathMapLayoutSpec

path = os.path.join(tempfile.mkdtemp(), "squares.nhydb")
with nhydb.create(path) as db:
    db.create_math_map_layout("squares", ("k",), u="k", V="k * k", side="NR",
                              payload_type="varint", layout_id=1)
    new = MathMap.compile("squares", ("k",), u="k", V="k * k + 1", side="NR")
    db.replace_math_spec(1, MathMapLayoutSpec("squares", new, 64))    # allowed: nothing written yet
    db.put_math(1, (12,), 144, commit=True)
    print(db.get_math(1, (12,)), db.get_math_cell(1, (12,))["_V_global"])
    try:
        db.replace_math_spec(1, MathMapLayoutSpec("squares", new, 64))
    except nhydb.MappingFrozenError:
        print("frozen")
    print(list(db.math_layouts))

144 145
frozen
[1]


*Expected output (from the cookbook):*

```text
144 145
frozen
[1]
```

NumPy input (2-D array, 1-D array or list of columns):

In [6]:
# requires: numpy
import os
import tempfile

import numpy as np

import nhydb
from nhydb.math import MathMap, MathMapLayoutSpec

path = os.path.join(tempfile.mkdtemp(), "mathnp.nhydb")
with nhydb.create(path) as db:
    m = MathMap.compile("g", ("x", "k"), u="x // 100 + 1", V="k * 10**100 + x % 100 + 1", side="x % 2")
    db.create_math_layout(MathMapLayoutSpec("g", m, 64), layout_id=1, payload_type="f64")
    x = np.arange(1000, dtype=np.int64)
    report = db.put_math_numpy(1, np.stack([x, x % 3], axis=1), x * 0.5, commit=True)
    print(report["cells"], db.get_math(1, (999, 0)))

1000 499.5


*Expected output (from the cookbook):*

```text
1000 499.5
```

Find MATH cells by input tuples with the QueryEngine provider
`math_layout_inputs`, and limit the size of intermediate results
(default 2^20 bits):

In [7]:
import os
import tempfile

import nhydb
from nhydb.math import MathExpression, MathMap, MathMapLayoutSpec, get_math_max_intermediate_bits, set_math_max_intermediate_bits
from nhydb.query import Q, QueryEngine

path = os.path.join(tempfile.mkdtemp(), "mathq.nhydb")
with nhydb.create(path) as db:
    grid = MathMap.compile("grid", ("x", "y"), u="x + 1", V="y", side="x % 2")
    db.create_math_layout(MathMapLayoutSpec("grid", grid, 64), layout_id=1, payload_type="utf8")
    db.put_math_batch(1, [((x, y), f"{x}:{y}") for x in (1, 2) for y in (1, 2)], commit=True)
    r = QueryEngine(db).execute(Q.math("math_layout_inputs", layout_id=1, inputs=[(1, 2), (2, 2), (7, 7)]),
                                return_mode="cell")
    print([(row["N"], row["value"]) for row in r["rows"]])

print(get_math_max_intermediate_bits())
e = MathExpression.compile("2 ** x", ("x",))
old = get_math_max_intermediate_bits()
set_math_max_intermediate_bits(64)
try:
    e.evaluate(x=100)
except nhydb.MathDomainError:
    print("too large")
finally:
    set_math_max_intermediate_bits(old)

[('N18', '1:2'), ('N22', '2:2'), ('N232', None)]
1048576
too large


*Expected output (from the cookbook):*

```text
[('N18', '1:2'), ('N22', '2:2'), ('N232', None)]
1048576
too large
```

There are two independent bit limits. Inside a database the limit comes
from `[math] math_max_intermediate_bits` of the configuration (default
2^20) and is **frozen into each MATH layout when it is created**
(`db.math_layouts[id].max_intermediate_bits`). `set_math_max_intermediate_bits`
changes only the process-wide limit for standalone `MathExpression`,
`MathMap` and `dry_run` objects, as in the example above. Neither limit
applies to values that a formula only passes through (`u="x"`): such
inputs may have any size.

**Very large numbers in Python output.** Python refuses to convert integers
with more than 4,300 decimal digits to text (`ValueError: Exceeds the limit
… for integer string conversion`). NHyDB stores and returns such numbers
exactly; only printing them needs `sys.set_int_max_str_digits(0)` (or the
environment variable `PYTHONINTMAXSTRDIGITS=0`, also for the command line).